# 06 Journalism Search And Retrieval Evaluation

Supplementary product appendix: structured vocabulary powers grounded table discovery. The search system returns source tables and evidence, not numeric answers.


In [1]:
import csv
import json
from pathlib import Path

import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent


def read_json(relative_path: str):
    path = ROOT / relative_path
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"missing": str(path)}


def csv_rows(relative_path: str, limit: int | None = None):
    csv.field_size_limit(2_147_483_647)
    path = ROOT / relative_path
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = list(csv.DictReader(file))
    return rows if limit is None else rows[:limit]


def csv_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    if not path.exists():
        return None
    return len(csv_rows(relative_path))


def parquet_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    return pq.read_table(path).num_rows if path.exists() else None

## Search Index


In [2]:
current = read_json("outputs/search/current_index.json")
summary_path = current.get("summary_path") or current.get("summary")
summary = read_json(summary_path) if summary_path else {}
{
    "current_index": current,
    "summary": summary,
}

{'current_index': {'index_manifest': 'outputs\\search\\run_c52b745f8105938306a7\\index_manifest.json',
  'lexical_index': 'outputs\\search\\run_c52b745f8105938306a7\\lexical.sqlite',
  'run_id': 'run_c52b745f8105938306a7',
  'search_documents': 'data\\processed\\search_documents.parquet',
  'semantic_index': 'outputs\\search\\run_c52b745f8105938306a7\\semantic.faiss',
  'semantic_metadata': 'outputs\\search\\run_c52b745f8105938306a7\\semantic_metadata.parquet'},
 'summary': {}}

## Retrieval Metrics


In [3]:
metrics = read_json("report/retrieval_metrics.json")
{
    "question_count": metrics.get("question_count"),
    "index_run_id": metrics.get("index", {}).get("run_id"),
    "selected_preset": metrics.get("tuning", {}).get("selected_preset"),
    "systems": metrics.get("systems", {}),
}

{'question_count': 200,
 'index_run_id': 'run_c52b745f8105938306a7',
 'selected_preset': 'geography_time_heavy',
 'systems': {'all_vocabulary_bm25': {'original': {'HitRate@1': 0.65,
    'HitRate@10': 0.95,
    'HitRate@5': 0.95,
    'MRR': 0.9333333333333333,
    'Relevance@1': 0.775,
    'Relevance@5': 0.975,
    'p50_latency_ms': 116.81810000300175,
    'p95_latency_ms': 275.4662999941502,
    'question_count': 20},
   'reformulated': {'HitRate@1': 0.25,
    'HitRate@10': 0.6,
    'HitRate@5': 0.6,
    'MRR': 0.658531746031746,
    'Relevance@1': 0.4,
    'Relevance@5': 0.7,
    'p50_latency_ms': 116.2696999963373,
    'p95_latency_ms': 275.39220000471687,
    'question_count': 20}},
  'fused': {'original': {'HitRate@1': 0.45,
    'HitRate@10': 0.95,
    'HitRate@5': 0.9,
    'MRR': 0.95,
    'Relevance@1': 0.775,
    'Relevance@5': 0.95,
    'p50_latency_ms': 317.6700999974855,
    'p95_latency_ms': 432.0934000061243,
    'question_count': 20},
   'reformulated': {'HitRate@1': 0.35,